# Programation Par Contraite - Covering Arrays avec contraintes semantiques

In [1]:
from ortools.sat.python import cp_model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import functools
from typing import Any, TypeAlias, Callable

## Qu'est ce qu'un Covering Array ?

> Les Covering Arrays (CA) sont des matrices $N \times k$ où chaque colonne représente un paramètre (facteur) avec $v$ valeurs, telles que toute combinaison de $t$ colonnes contienne tous les $t$-uplets possibles au moins une fois.

Dit comme cela, la définition peut sembler un peu abstraite, alors voyons un exemple extrêmement simple.

Imaginons un formulaire où nous demandons la majorité du participant (Majeur ou non), son statut professionnel (Employé ou non) et sa nationalité (Française ou non). Nous avons donc 3 paramètres ($k = 3$), et chacun peut prendre exactement 2 valeurs ($v = 2$).

En théorie, il y a $2 \times 2 \times 2 = 2^3 = v^k = 8$ combinaisons possibles si on voulait tout tester. L'idée du Covering Array est de ne pas tester toutes les combinaisons globales, mais de garantir la couverture sur un groupe de $t$ colonnes. Si on prend une force $t = 2$ (*pairwise testing*), on obtient une matrice où chaque paire de colonnes contient toutes les paires de valeurs possibles au moins une fois.

$$
\text{CA} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0 \\
1 & 1 & 1 
\end{bmatrix}
$$

Ici, on peut voir que si on isole deux colonnes au choix (par exemple *Majeur* et *Français*), on retrouve bien les 4 paires possibles : $(0, 0)$, $(0, 1)$, $(1, 0)$ et $(1, 1)$. 

Tous les cas globaux n'ont pas été testés (seulement 4 lignes sur 8), mais cette méthode permet de réduire grandement le nombre de tests tout en gardant une excellente capacité à détecter les bugs d'interactions.

## Contraintes Sémentiques

Vous avez peut-être relevé un problème logique dans l'exemple précédent. En effet, la troisième ligne propose un profil **Mineur (0) et Employé (1)**, ce qui est interdit par la loi (plus ou moins). 

Pour résoudre ce problème, on ajoute des **contraintes sémantiques** exprimées en logique propositionnelle : ici, $\neg(\text{Mineur} \land \text{Employé})$. 

Exclure des combinaisons rend le problème plus difficile pour le solveur. Comme certaines lignes "partagées" deviennent interdites, le solveur doit réorganiser la matrice et est parfois obligé d'**augmenter le nombre de lignes $N$** pour réussir à couvrir le reste des paires légitimes.

Avec notre contrainte, le nombre minimal de lignes passe de $N=4$ à $N=5$ :

$$
\text{CA}_{\text{contraint}} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
1 & 1 & 0 \\
1 & 0 & 1 \\
0 & 0 & 0 \\
0 & 0 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

On remarque qu'aucune ligne ne contient la combinaison interdite $(0, 1)$ sur les colonnes Majeur/Employé, mais que toutes les autres paires du système restent testées.

## Sujet

Dans ce projet, nous allons nous intéresser a la génération de Covering Array avec contraintes sémentiques et une force $t \ge 3$.

In [2]:
Parameter: TypeAlias = str
Value: TypeAlias = Any
Values: TypeAlias = list[Value]
Parameters: TypeAlias = dict[Parameter, Values]
ParameterMap: TypeAlias = dict[Parameter, dict[Value, int]]
Constraint: TypeAlias = Callable[[cp_model.CpModel, dict[Parameter, cp_model.IntVar], ParameterMap], None]

In [3]:
class CoveringArray:
    def __init__(self, matrix: list[list[int]], parameters: Parameters) -> None:
        self.matrix = matrix
        self.parameters = parameters

    def get_matrix(self) -> list[list[int]]:
        return self.matrix

    def to_human_readable(self) -> list[dict[Parameter, Value]]:
        return [
            {
                parameter_name: self.parameters[parameter_name][row[column_idx]]
                for column_idx, parameter_name in enumerate(self.parameters.keys())
            }
            for row in self.matrix
        ]

    def pretty_print(self) -> None:
        df = pd.DataFrame(self.to_human_readable())
        print(df)        

In [6]:
class CoveringArrayBuilder:
    def __init__(self, parameters: Parameters, t_force: int = 3) -> None:
        self.parameters = parameters
        self.t_force = t_force

        self.k: int = len(parameters)

        assert self.k >= self.t_force, f"We can't have less parameters than t force, got {self.k} parameters but {self.t_force} t force"
    
        self.constraints: list[Constraint] = []

        self.parameter_map: ParameterMap = {
            parameter: {value: index for index, value in enumerate(values)}
            for parameter, values in parameters.items()
        }

    def add_constraint(self, constraint: Constraint) -> "CoveringArrayBuilder":
        self.constraints.append(constraint)
        return self

    def _build_model(self) -> cp_model.CpModel:
        model = cp_model.CpModel()

        v = [len(values) for values in self.parameters.values()]
        N_max: int = functools.reduce(lambda x, y: x * y, v)

        # Gives the solver a matrix of all possible values for each variables
        x: list[dict[Parameter, cp_model.IntVar]] = [
            {
                parameter: model.new_int_var(0, len(values) - 1, f"x_{index}_{parameter}")
                for parameter, values in self.parameters.items()
            }
            for index in range(N_max)
        ]

        # Forces the solver to use rows in order
        row_is_used = [model.new_bool_var(f"row_used_{index}") for index in range(N_max)]
        for index in range(N_max - 1):
            model.add_implication(row_is_used[index + 1], row_is_used[index])

        # Applies the different user defined constraints to the model
        for row in x:
            for constraint in self.constraints:
                constraint(model, row, self.parameter_map)

        # Forces each variable possible value to appear at least once
        for parameter, values in self.parameters.items():
            for value in values:
                matches = []
                
                for index, row in enumerate(x):
                    b = model.new_bool_var(f"has_{parameter}_{value}_row{index}")
                    model.add(row[parameter] == self.parameter_map[parameter][value]).only_enforce_if(b)
                    model.add(row[parameter] != self.parameter_map[parameter][value]).only_enforce_if(b.Not())
        
                    matches.append(b)

                model.add_bool_or(matches)

        # Generate all t-wise interactions
        for parameter_subset in itertools.combinations(self.parameters.keys(), self.t_force):
            domains = [self.parameters[parameter] for parameter in parameter_subset]
    
            for values_tuple in itertools.product(*domains):
                # If the interaction is impossible due to user defined constraits we don't include it
                if not self._is_interaction_feasible(parameter_subset, values_tuple):
                    continue
    
                # Create "row matches interaction" booleans
                matching_rows = []
    
                for row_index, row in enumerate(x):
                    row_matches = model.new_bool_var(
                        f"match_{row_index}_{'_'.join(parameter_subset)}_{'_'.join(map(str, values_tuple))}"
                    )
    
                    conditions = []
    
                    # row[param] == wanted_value
                    for parameter, value in zip(parameter_subset, values_tuple):
                        b = model.new_bool_var(f"eq_{row_index}_{parameter}_{value}")
                        model.add(row[parameter] == self.parameter_map[parameter][value]).only_enforce_if(b)
                        model.add(row[parameter] != self.parameter_map[parameter][value]).only_enforce_if(b.Not())
    
                        conditions.append(b)
    
                    # row_matches <=> all conditions true
                    model.add_bool_and(conditions).only_enforce_if(row_matches)

                    # if row_matches then row must be used
                    model.add_implication(row_matches, row_is_used[row_index])
    
                    matching_rows.append(row_matches)
    
                # Coverage constraint: at least one row matches this interaction
                model.add_bool_or(matching_rows)
    
        # Objective: minimize N the number of rows used
        model.minimize(sum(row_is_used))

        return model, x, row_is_used

    def _is_interaction_feasible(self, parameter_subset, values_tuple) -> bool:
        test_model = cp_model.CpModel()

        # Create one fake row
        row = {
            parameter: test_model.new_int_var(0, len(self.parameters[parameter]) - 1, parameter)
            for parameter in self.parameters
        }
    
        # Apply all user constraints
        for constraint in self.constraints:
            constraint(test_model, row, self.parameter_map)
    
        # Force the tested interaction
        for param, value in zip(parameter_subset, values_tuple):
            test_model.add(row[param] == self.parameter_map[param][value])
    
        # Check satisfiable
        solver = cp_model.CpSolver()
    
        status = solver.solve(test_model)
    
        return status in (cp_model.FEASIBLE, cp_model.OPTIMAL)

    def solve(self, max_time_seconds: float = 10.0) -> CoveringArray:
        model, x, row_is_used = self._build_model()
        
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = max_time_seconds
    
        status = solver.solve(model)
    
        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            raise RuntimeError("No covering array found")
    
        matrix = [
            [solver.value(row_variables[parameter]) for parameter in self.parameters]
            for row_index, row_variables in enumerate(x)
            if solver.value(row_is_used[row_index])
        ]
    
        return CoveringArray(matrix, self.parameters)


In [7]:
mon_systeme = {
    "Majorité": ["Mineur", "Majeur"],
    "Emploi": ["Chômage", "Travailleur"],
    "Nationalité": ["Française", "Étrangère"],
    "Bob": ["Dylan", "Marley"],
}

def c(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    is_minor = model.new_bool_var("is_minor")
    model.add(row["Majorité"] == parameter_map["Majorité"]["Mineur"]).only_enforce_if(is_minor)
    model.add(row["Majorité"] == parameter_map["Majorité"]["Majeur"]).only_enforce_if(is_minor.Not())
    
    model.add(row["Emploi"] != parameter_map["Emploi"]["Travailleur"]).only_enforce_if(is_minor)

CA_builder = CoveringArrayBuilder(mon_systeme)
CA_builder.add_constraint(c)
CA = CA_builder.solve()
CA.get_matrix()

[[1, 1, 1, 1],
 [1, 1, 1, 0],
 [1, 1, 0, 1],
 [1, 1, 0, 0],
 [0, 0, 1, 1],
 [0, 0, 1, 0],
 [0, 0, 0, 1],
 [0, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 0, 0]]

In [13]:
parameters = {
    "AgeGroup": ["Child", "Teen", "Adult", "Senior"],
    "License": ["Yes", "No"],
    "Employment": ["Employed", "Unemployed"],
    "Driving": ["Allowed", "NotAllowed"],
}

def child_cannot_have_licence_or_work_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    is_child = model.new_bool_var("is_child")
    model.add(row["AgeGroup"] == parameter_map["AgeGroup"]["Child"]).only_enforce_if(is_child)
    model.add(row["AgeGroup"] != parameter_map["AgeGroup"]["Child"]).only_enforce_if(is_child.Not())
    
    model.add(row["License"] != parameter_map["License"]["Yes"]).only_enforce_if(is_child)
    model.add(row["Employment"] != parameter_map["Employment"]["Employed"]).only_enforce_if(is_child)

def seniors_cannot_work_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    is_senior = model.new_bool_var("is_senior")
    model.add(row["AgeGroup"] == parameter_map["AgeGroup"]["Senior"]).only_enforce_if(is_senior)
    model.add(row["AgeGroup"] != parameter_map["AgeGroup"]["Senior"]).only_enforce_if(is_senior.Not())
    
    model.add(row["Employment"] != parameter_map["Employment"]["Employed"]).only_enforce_if(is_senior)

def need_licence_to_drive_constraint(model: cp_model.CpModel, row: dict[Parameter, cp_model.IntVar], parameter_map: ParameterMap) -> None:
    no_license = model.new_bool_var("no_license")
    model.add(row["License"] == parameter_map["License"]["No"]).only_enforce_if(no_license)
    model.add(row["License"] != parameter_map["License"]["No"]).only_enforce_if(no_license.Not())

    model.Add(row["Driving"] != parameter_map["Driving"]["Allowed"]).only_enforce_if(no_license)

CA_builder = CoveringArrayBuilder(parameters)
CA_builder.add_constraint(child_cannot_have_licence_or_work_constraint)
CA_builder.add_constraint(seniors_cannot_work_constraint)
CA_builder.add_constraint(need_licence_to_drive_constraint)
CA = CA_builder.solve()
CA.pretty_print()

   AgeGroup License  Employment     Driving
0    Senior      No  Unemployed  NotAllowed
1     Adult      No    Employed  NotAllowed
2     Adult     Yes  Unemployed  NotAllowed
3    Senior     Yes  Unemployed     Allowed
4      Teen     Yes    Employed  NotAllowed
5     Adult     Yes    Employed     Allowed
6     Adult     Yes  Unemployed     Allowed
7      Teen      No  Unemployed  NotAllowed
8      Teen     Yes  Unemployed     Allowed
9      Teen     Yes    Employed     Allowed
10    Child      No  Unemployed  NotAllowed
11   Senior     Yes  Unemployed  NotAllowed
12    Adult      No  Unemployed  NotAllowed
13     Teen      No    Employed  NotAllowed
